<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 07 — Support Vector Machine (SVM)
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Margens, Kernels e Terceiro Modelo no Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">⚔️ 3º Modelo do Curso</span>
</div>


## Onde estamos?

Nos últimos notebooks treinamos dois modelos no Titanic:

| Aula | Modelo | Como classifica | Interpretável? |
|------|--------|-----------------|---------------|
| 05 | **KNN** | Voto dos K vizinhos mais próximos | Baixa |
| 06 | **Regressão Logística** | Probabilidade via sigmoide | Alta — coeficientes |
| **07** | **SVM** | **Hiperplano de margem máxima** | Média |

Hoje aprendemos o **SVM (Support Vector Machine)**. Enquanto o KNN pergunta
*"quem são os mais parecidos?"* e a Regressão Logística pergunta *"qual a probabilidade?",*
o SVM pergunta:

> *"Qual é a fronteira que separa as classes com a maior distância possível?"*

Essa ideia de **maximizar a margem** torna o SVM robusto e eficaz — especialmente
quando os dados têm muitas features ou as fronteiras não são simples retas.

---

## Roteiro de hoje

| Parte | Tema |
|-------|------|
| **Config** | Recarregando o Titanic e os modelos anteriores |
| **1** | Intuição — o que é um hiperplano de margem máxima? | 
| **2** | Margem rígida vs suave — o parâmetro C | 
| **3** | O Kernel Trick — dados não-linearmente separáveis | 
| **4** | Nosso terceiro modelo no Titanic | 
| **5** | Ajustando C e o kernel — impacto nas métricas | 
| **6** | KNN vs Log. Reg. vs SVM — placar geral | 

<div style="background:#d1ecf1; border-left:5px solid #0c5460; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#0c5460;">ℹ️ </strong><span style="color:#0c5460;">O SVM é especialmente poderoso quando as classes <strong>não são linearmente separáveis</strong> — situação muito comum na prática. O Kernel Trick (Parte 3) é um dos truques matemáticos mais elegantes do ML.</span></div>

---

## Configuração — Execute antes de começar


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

# ── Pipeline completo das aulas anteriores ────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

df = sns.load_dataset("titanic").copy()
df["age"]       = df["age"].fillna(df["age"].median())
df["embarked"]  = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"]  = df["sibsp"] + df["parch"] + 1
df["sozinho"]          = (df["tamanho_familia"] == 1).astype(int)
df["faixa_etaria_enc"] = pd.cut(df["age"], bins=[0,12,18,60,100],
                                 labels=[0,1,2,3]).astype(int)
df["titulo"]           = df["name"].str.extract(r",\s([A-Za-z]+)\.")                            .iloc[:,0].map(lambda t: t if t in
                           ["Mr","Miss","Mrs","Master"] else "Raro")
df["tarifa_por_pessoa"]= (df["fare"] / df["tamanho_familia"].clip(lower=1)).round(2)
df["sex_enc"]          = (df["sex"] == "female").astype(int)
df["pclass_enc"]       = df["pclass"].map({1:2, 2:1, 3:0})

embarked_ohe = pd.get_dummies(df["embarked"], prefix="embarked", drop_first=True)
titulo_ohe   = pd.get_dummies(df["titulo"],   prefix="titulo",   drop_first=True)
df = pd.concat([df, embarked_ohe, titulo_ohe], axis=1)

FEATURES = ["pclass_enc","sex_enc","age","tamanho_familia","sozinho",
            "faixa_etaria_enc","tarifa_por_pessoa","embarked_q","embarked_s",
            "titulo_Master","titulo_Miss","titulo_Mr","titulo_Mrs","titulo_Raro"]
FEATURES = [f for f in FEATURES if f in df.columns]

X = df[FEATURES].copy()
y = df["survived"].copy()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)
X_teste_sc  = scaler.transform(X_teste)

# Modelos das aulas anteriores para comparação final
knn_ref = KNeighborsClassifier(n_neighbors=7).fit(X_treino_sc, y_treino)
lr_ref  = LogisticRegression(max_iter=1000, random_state=42).fit(X_treino_sc, y_treino)

def resumo_modelo(nome, modelo, Xte, yte):
    yp  = modelo.predict(Xte)
    ypr = modelo.predict_proba(Xte)[:,1] if hasattr(modelo,"predict_proba") else None
    auc = roc_auc_score(yte, ypr) if ypr is not None else float("nan")
    return {"Modelo": nome,
            "Acurácia": accuracy_score(yte, yp),
            "F1":       f1_score(yte, yp),
            "AUC-ROC":  auc}

print("✅ Dataset e modelos de referência prontos!")
print(f"   Treino: {X_treino_sc.shape[0]} amostras | Teste: {X_teste_sc.shape[0]} amostras")
print()
print("Desempenho dos modelos anteriores (referência):")
for r in [resumo_modelo("KNN (K=7)", knn_ref, X_teste_sc, y_teste),
          resumo_modelo("Reg. Logística", lr_ref, X_teste_sc, y_teste)]:
    print(f"  {r['Modelo']:<18} Acurácia={r['Acurácia']:.1%}  F1={r['F1']:.4f}  AUC={r['AUC-ROC']:.4f}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Intuição — O Hiperplano de Margem Máxima</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Não basta separar as classes — queremos separar com a maior folga possível."</p></div></div></div>


### O problema fundamental da classificação

Imagine dois grupos de pontos num plano.
É possível traçar infinitas retas que os separem.
Qual é a **melhor** reta?

O SVM responde: a reta (ou hiperplano em N dimensões) que **maximiza
a distância até os pontos mais próximos de cada classe**.
Essa distância é chamada de **margem**.

```
         Classe 0 (não sobreviveu)          Classe 1 (sobreviveu)
         ●  ●                                       ○  ○
            ●    ●                             ○  ○
                    ●   ← margem →  ○
                       ║══════════║
                       ↑
               Hiperplano ótimo
               (equidistante dos pontos mais próximos)
```

### Por que maximizar a margem?

Uma margem maior significa que o modelo tem **mais tolerância a erros** em dados novos.
É como construir uma estrada com acostamento generoso — se um carro sair um pouco
da pista, não cai no barranco imediatamente.

### Vocabulário essencial

| Termo | Significado |
|-------|-------------|
| **Hiperplano** | A fronteira de decisão — uma reta em 2D, um plano em 3D, um hiperplano em ND |
| **Vetores de Suporte** | Os pontos **mais próximos** do hiperplano — são eles que definem a margem |
| **Margem** | A distância entre o hiperplano e os vetores de suporte de cada classe |
| **Margem Máxima** | O objetivo do SVM: encontrar o hiperplano com maior margem possível |

<div style="background:#e8d5f5; border-left:5px solid #5b2c8d; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#5b2c8d;">💡 </strong><span style="color:#5b2c8d;">O nome <strong>Support Vector Machine</strong> vem dos <em>vetores de suporte</em>: se você remover qualquer outro ponto do dataset e re-treinar o SVM, o hiperplano permanece o mesmo. Só os vetores de suporte importam para definir a fronteira.</span></div>


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Observe o diagrama abaixo e responda antes de rodar o código: (a) por que a Reta B é melhor que a Reta A mesmo que ambas separem as classes? (b) o que acontece se um novo ponto aparecer perto da fronteira? Qual reta é mais robusta? Registre sua resposta abaixo.</span></div>

*✏️ (a) A Reta B é melhor porque: `???`*

*✏️ (b) Para novos pontos próximos à fronteira, a Reta `???` é mais robusta porque: `???`*


In [ ]:
# Visualizando o conceito de margem máxima
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

# Dois grupos linearmente separáveis
X_neg = np.random.randn(20, 2) + np.array([-2.0, -1.0])
X_pos = np.random.randn(20, 2) + np.array([ 2.0,  1.0])
X_vis = np.vstack([X_neg, X_pos])
y_vis = np.array([0]*20 + [1]*20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Por que Maximizar a Margem?", fontsize=13, fontweight="bold")

x_line = np.linspace(-5, 5, 200)

for ax, titulo, retas in [
    (axes[0], "Muitas retas separam as classes Qual escolher?",
     [(-0.3, 0.2,  "#aaa", "--", "Reta A (funciona)"),
      ( 0.5, 0.1,  "#888", "-.", "Reta B (funciona)"),
      ( 0.0, 0.5,  "#bbb", ":",  "Reta C (funciona)")]),
    (axes[1], "SVM escolhe a de MARGEM MÁXIMA
(máxima distância aos pontos mais próximos)",
     [( 0.5, 0.0, "#0f3460", "-",  "Hiperplano ótimo"),
      ( 0.5, 1.2, "#0f3460", "--", "Margem superior"),
      ( 0.5,-1.2, "#0f3460", "--", "Margem inferior")])
]:
    # Pontos
    ax.scatter(X_neg[:,0], X_neg[:,1], c="#e94560", s=60, edgecolors="white",
               linewidth=0.8, label="Classe 0", zorder=5)
    ax.scatter(X_pos[:,0], X_pos[:,1], c="#0f3460", s=60, edgecolors="white",
               linewidth=0.8, label="Classe 1", zorder=5)

    for b0, b1, cor, ls, lbl in retas:
        ax.plot(x_line, b0*x_line + b1, color=cor, linestyle=ls,
                linewidth=2, label=lbl, alpha=0.85)

    if titulo.startswith("SVM"):
        ax.fill_between(x_line, 0.5*x_line - 1.2, 0.5*x_line + 1.2,
                         alpha=0.06, color="#0f3460", label="Zona da margem")
        # Destaque nos vetores de suporte (pontos mais próximos)
        sv_neg = X_neg[np.argmin(X_neg[:,0])]
        sv_pos = X_pos[np.argmin(X_pos[:,0])]
        for sv, cor in [(sv_neg,"#e94560"),(sv_pos,"#0f3460")]:
            ax.scatter(*sv, s=180, facecolors="none", edgecolors=cor,
                       linewidth=2.5, zorder=10)
        ax.annotate("Vetor de Supporte", sv_neg,
                    xytext=(sv_neg[0]-1.8, sv_neg[1]+0.8),
                    arrowprops=dict(arrowstyle="->", color="#333"),
                    fontsize=8, color="#333")

    ax.set_xlim(-5.5, 5.5); ax.set_ylim(-5, 5)
    ax.set_title(titulo, fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_xlabel("Feature 1"); ax.set_ylabel("Feature 2")

plt.tight_layout()
plt.savefig("aula07_margem_maxima.png", dpi=110, bbox_inches="tight")
plt.show()


In [ ]:
# ── GABARITO DA MISSÃO 1 (descomente para ver) ───────────────────────────────
# print("Gabarito:")
# print()
# print("(a) A Reta B (hiperplano de margem maxima) e melhor porque:")
# print("    Ela esta equidistante dos pontos mais proximos de cada classe.")
# print("    As outras retas ficam muito perto de um dos grupos,")
# print("    deixando pouca 'folga' para erros em dados novos.")
# print()
# print("(b) Para novos pontos proximos a fronteira:")
# print("    A reta de margem maxima e mais robusta porque tem o maior espaco")
# print("    de seguranca. Um novo ponto pode estar um pouco errado (ruido,")
# print("    medicao imprecisa) e ainda sera classificado corretamente.")
# print("    Retas com margem pequena erram com qualquer variacao minima.")
# print()
# print("Analogia: e como estacionar um carro numa vaga apertada vs. folgada.")
# print("Na vaga folgada, errar 5cm nao causa problema. Na apertada, sim.")


### A matemática da margem (nível intuitivo)

O SVM encontra o hiperplano resolvendo um problema de otimização:

```
Maximizar:  2 / ||w||        ← largura da margem

Sujeito a:
  wᵀxᵢ + b ≥ +1   para todos os pontos da Classe 1
  wᵀxᵢ + b ≤ −1   para todos os pontos da Classe 0
```

Onde:
- `w` é o vetor normal ao hiperplano (define a direção)
- `b` é o viés (define a posição)
- `||w||` é a norma do vetor — minimizá-la equivale a maximizar a margem

Você não precisa resolver isso manualmente — o scikit-learn faz isso.
Mas entender a intuição é fundamental para saber quando e como usar o SVM.


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Margem Rígida vs Suave — O Parâmetro C</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Na vida real, os dados têm ruído. O SVM precisa ser tolerante."</p></div></div></div>


### O problema da margem rígida

A formulação da Parte 1 é chamada de **margem rígida (hard margin)**:
nenhum ponto pode cruzar a margem — a separação tem que ser perfeita.

**Problema:** na prática, dados reais têm ruído, outliers e sobreposição entre classes.
A margem rígida falha completamente nesses casos.

### A solução: margem suave (soft margin)

O SVM com margem suave permite que **alguns pontos violem a margem** ou até
estejam do lado errado da fronteira — mas cobra um custo por isso.

O **parâmetro C** controla esse trade-off:

```
C grande  →  punição alta por erros  →  margem menor, menos violações
             (mais rígido, tenta acertar tudo → risco de overfitting)

C pequeno →  punição baixa por erros →  margem maior, mais violações
             (mais tolerante → melhor generalização)
```

<div style="background:#fff3cd; border-left:5px solid #856404; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#856404;">⚠️ </strong><span style="color:#856404;">C é o hiperparâmetro mais importante do SVM. Um C muito grande faz o modelo decorar o treino (overfitting). Um C muito pequeno ignora padrões importantes (underfitting). O valor correto deve ser encontrado por validação cruzada.</span></div>


In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_classification

np.random.seed(42)

# Dataset 2D com alguma sobreposição entre classes
X_2d, y_2d = make_classification(
    n_samples=150, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1,
    class_sep=0.7, random_state=42
)
# Adicionando outliers propositais
X_2d = np.vstack([X_2d, [[-0.5, 2.5], [0.8, -2.5]]])
y_2d = np.concatenate([y_2d, [1, 0]])

X_2d_tr, X_2d_te, y_2d_tr, y_2d_te = train_test_split(
    X_2d, y_2d, test_size=0.3, random_state=42)

# Testando diferentes valores de C
valores_C = [0.01, 0.5, 5.0, 100.0]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Efeito do Parâmetro C — Margem Rígida vs Suave",
             fontsize=13, fontweight="bold")

h = 0.05
for ax, C_val in zip(axes, valores_C):
    svm_c = SVC(kernel="linear", C=C_val, random_state=42)
    svm_c.fit(X_2d_tr, y_2d_tr)

    # Fronteira de decisão
    x_min, x_max = X_2d[:,0].min()-0.5, X_2d[:,0].max()+0.5
    y_min, y_max = X_2d[:,1].min()-0.5, X_2d[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
    Z = svm_c.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.20, cmap="RdBu")
    ax.contour(xx, yy, Z, colors="gray", linewidths=0.5, alpha=0.4)

    # Margem
    if hasattr(svm_c, "decision_function"):
        Z_df = svm_c.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contour(xx, yy, Z_df, levels=[-1, 0, 1],
                   linestyles=["--","-","--"],
                   colors=["#e94560","#0f3460","#0f3460"],
                   linewidths=[1.5, 2.5, 1.5])

    # Pontos
    ax.scatter(X_2d_tr[:,0], X_2d_tr[:,1], c=y_2d_tr,
               cmap="RdBu", edgecolors="white", s=40, linewidth=0.6, zorder=5)

    # Vetores de suporte
    ax.scatter(svm_c.support_vectors_[:,0], svm_c.support_vectors_[:,1],
               s=180, facecolors="none", edgecolors="#f0a500",
               linewidth=2, zorder=10, label=f"SVs: {len(svm_c.support_vectors_)}")

    acc = svm_c.score(X_2d_te, y_2d_te)
    titulo = "Underfitting" if C_val < 0.1 else ("Overfitting" if C_val > 50 else "Equilibrado")
    ax.set_title(f"C = {C_val} Acurácia teste: {acc:.0%} ({titulo})",
                 fontsize=9, fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")
    ax.set_xlabel("F1"); ax.set_ylabel("F2")

plt.tight_layout()
plt.savefig("aula07_parametro_C.png", dpi=110, bbox_inches="tight")
plt.show()

print("Observe o número de vetores de suporte (SVs) em cada gráfico:")
print("  C pequeno → mais SVs → margem mais larga → mais tolerante")
print("  C grande  → menos SVs → margem mais estreita → mais rígido")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Olhando para os 4 gráficos gerados, responda: (a) com C=0.01, o modelo está overfittando ou underfittando? Como você sabe? (b) com C=100, o que acontece com a margem e com os vetores de suporte? (c) qual C você escolheria como ponto de partida e por quê?</span></div>

*✏️ (a) C=0.01: o modelo está `???` porque: `???`*

*✏️ (b) C=100: margem `???`, vetores de suporte `???`*

*✏️ (c) Escolheria C=`???` porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# print("Gabarito:")
# print()
# print("(a) C=0.01 — UNDERFITTING:")
# print("    A margem e muito larga, aceitando muitas violacoes.")
# print("    O modelo ignora detalhes importantes dos dados.")
# print("    Sinal: acuracia baixa tanto no treino quanto no teste,")
# print("    fronteira de decisao muito simples.")
# print()
# print("(b) C=100 — Margem ESTREITA, poucos Vetores de Suporte:")
# print("    A fronteira se dobra para tentar acertar todos os pontos,")
# print("    incluindo os outliers. Poucos pontos estao na margem.")
# print("    Risco de overfitting em dados novos.")
# print()
# print("(c) Ponto de partida recomendado: C=1.0 (padrao do sklearn)")
# print("    E um valor conservador que equilibra rigidez e tolerancia.")
# print("    Depois ajustamos com validacao cruzada (Grid Search).")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">O Kernel Trick — Dados Não-Linearmente Separáveis</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Quando uma reta não resolve, vamos para uma dimensão superior."</p></div></div></div>


### O problema da não-linearidade

Muitos problemas reais não podem ser separados por uma reta.
O exemplo clássico: dois grupos concêntricos.

A sacada do SVM é o **Kernel Trick**: em vez de transformar
os dados explicitamente para uma dimensão maior (caro computacionalmente),
o kernel calcula diretamente o produto interno nessa dimensão superior.

> *"É como resolver um labirinto mais fácil no 3D do que no 2D."*

### Kernels disponíveis

| Kernel | Fronteira | Parâmetros | Quando usar |
|--------|-----------|------------|-------------|
| **linear** | Reta / hiperplano | C | Dados linearmente separáveis; muitas features (NLP) |
| **rbf** | Curva gaussiana | C, gamma | Padrão — funciona bem na maioria dos problemas |
| **poly** | Fronteira polinomial | C, degree | Quando a relação é polinomial conhecida |
| **sigmoid** | Similar à rede neural | C, coef0 | Raramente usado |

### O parâmetro gamma (kernel RBF)

```
gamma grande  →  cada ponto tem influência LOCAL  →  fronteira irregular  →  overfitting
gamma pequeno →  cada ponto tem influência GLOBAL →  fronteira suave     →  underfitting
```

<div style="background:#e8d5f5; border-left:5px solid #5b2c8d; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#5b2c8d;">💡 </strong><span style="color:#5b2c8d;">Regra prática: comece sempre com <code>kernel='rbf'</code> e <code>C=1.0</code>. Se os dados forem de texto ou tiverem muitas features esparsas, use <code>kernel='linear'</code>.</span></div>


In [ ]:
from sklearn.datasets import make_circles, make_moons

np.random.seed(42)

# Três datasets desafiadores para retas
datasets = [
    ("Circulos (reta impossivel)",
     make_circles(n_samples=150, noise=0.07, factor=0.4, random_state=42)),
    ("Luas (reta impossivel)",
     make_moons(n_samples=150, noise=0.1, random_state=42)),
    ("Blobs com ruido (reta dificil)",
     make_classification(n_samples=150, n_features=2, n_redundant=0,
                         n_informative=2, n_clusters_per_class=2,
                         class_sep=0.5, random_state=42)),
]

kernels = [("linear", {"C":1.0}), ("rbf", {"C":1.0, "gamma":"scale"})]
fig, axes = plt.subplots(len(datasets), len(kernels), figsize=(12, 14))
fig.suptitle("Kernel Trick — Linear vs RBF em Dados Nao-Lineares",
             fontsize=13, fontweight="bold")

h = 0.04
for row, (nome_ds, (X_d, y_d)) in enumerate(datasets):
    X_d_sc = StandardScaler().fit_transform(X_d)
    X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
        X_d_sc, y_d, test_size=0.25, random_state=42)

    for col, (kernel_nome, params) in enumerate(kernels):
        ax = axes[row, col]
        svm_k = SVC(kernel=kernel_nome, **params, random_state=42)
        svm_k.fit(X_tr_d, y_tr_d)
        acc = svm_k.score(X_te_d, y_te_d)

        x_min, x_max = X_d_sc[:,0].min()-0.3, X_d_sc[:,0].max()+0.3
        y_min, y_max = X_d_sc[:,1].min()-0.3, X_d_sc[:,1].max()+0.3
        xx, yy = np.meshgrid(np.arange(x_min,x_max,h),
                              np.arange(y_min,y_max,h))
        Z = svm_k.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.25, cmap="RdBu")
        ax.scatter(X_d_sc[:,0], X_d_sc[:,1], c=y_d,
                   cmap="RdBu", edgecolors="white", s=35,
                   linewidth=0.5, zorder=5)

        titulo_col = "SVM Linear" if col==0 else "SVM RBF"
        ok = "✅" if acc > 0.85 else ("⚠️" if acc > 0.7 else "❌")
        ax.set_title(f"{titulo_col} {nome_ds} Acurácia: {acc:.0%} {ok}",
                     fontsize=9, fontweight="bold")
        ax.set_xlabel("F1"); ax.set_ylabel("F2")
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig("aula07_kernels.png", dpi=110, bbox_inches="tight")
plt.show()

print("Conclusao: o kernel RBF consegue aprender fronteiras nao-lineares!")
print("O kernel linear falha nos dados circulares e de luas.")


In [ ]:
# Visualizando o Kernel Trick: de 2D para 3D
from mpl_toolkits.mplot3d import Axes3D

np.random.seed(42)
X_circ, y_circ = make_circles(n_samples=100, noise=0.05, factor=0.4, random_state=42)

# A transformacao do kernel RBF: phi(x) = (x1, x2, x1² + x2²)
X_3d = np.column_stack([X_circ, X_circ[:,0]**2 + X_circ[:,1]**2])

fig = plt.figure(figsize=(14, 5))

# Vista 2D: nao separavel
ax1 = fig.add_subplot(131)
ax1.scatter(X_circ[y_circ==0, 0], X_circ[y_circ==0, 1],
            c="#e94560", s=40, edgecolors="white", alpha=0.8)
ax1.scatter(X_circ[y_circ==1, 0], X_circ[y_circ==1, 1],
            c="#0f3460", s=40, edgecolors="white", alpha=0.8)
ax1.set_title("2D — Nao separavel por reta", fontweight="bold")
ax1.set_xlabel("X1"); ax1.set_ylabel("X2")

# Vista 3D: separavel por plano
ax2 = fig.add_subplot(132, projection="3d")
ax2.scatter(X_3d[y_circ==0, 0], X_3d[y_circ==0, 1], X_3d[y_circ==0, 2],
            c="#e94560", s=40, alpha=0.8)
ax2.scatter(X_3d[y_circ==1, 0], X_3d[y_circ==1, 1], X_3d[y_circ==1, 2],
            c="#0f3460", s=40, alpha=0.8)
ax2.set_title("3D apos transformacao (separavel por plano!)", fontweight="bold")
ax2.set_xlabel("X1"); ax2.set_ylabel("X2"); ax2.set_zlabel("X1²+X2²")

# Plano separador no espaco 3D
xx_p = np.linspace(-1.5, 1.5, 20)
yy_p = np.linspace(-1.5, 1.5, 20)
xx_p, yy_p = np.meshgrid(xx_p, yy_p)
zz_p = np.full_like(xx_p, 0.25)
ax2.plot_surface(xx_p, yy_p, zz_p, alpha=0.15, color="#f0a500")

# Separacao perfeita no 3D
ax3 = fig.add_subplot(133)
ax3.scatter(X_circ[y_circ==0, 0], X_circ[y_circ==0, 1],
            c="#e94560", s=40, edgecolors="white", alpha=0.8,
            label="Classe 0")
ax3.scatter(X_circ[y_circ==1, 0], X_circ[y_circ==1, 1],
            c="#0f3460", s=40, edgecolors="white", alpha=0.8,
            label="Classe 1")
svm_rbf_demo = SVC(kernel="rbf", C=1.0).fit(X_circ, y_circ)
h_d = 0.04
xm, xM = X_circ[:,0].min()-0.3, X_circ[:,0].max()+0.3
ym, yM = X_circ[:,1].min()-0.3, X_circ[:,1].max()+0.3
gx, gy = np.meshgrid(np.arange(xm,xM,h_d), np.arange(ym,yM,h_d))
GZ = svm_rbf_demo.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
ax3.contourf(gx, gy, GZ, alpha=0.2, cmap="RdBu")
ax3.contour(gx, gy, GZ, colors="gray", linewidths=0.8, alpha=0.5)
ax3.set_title("SVM RBF no 2D original\n(fronteira circular aprendida!)",
              fontweight="bold")
ax3.legend(fontsize=8)
ax3.set_xlabel("X1"); ax3.set_ylabel("X2")

plt.suptitle("O Kernel Trick — Transformando o Espaco para Separar as Classes",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("aula07_kernel_trick_3d.png", dpi=110, bbox_inches="tight")
plt.show()

print("O Kernel Trick calcula o produto interno no espaco 3D")
print("SEM transformar os dados explicitamente — economizando muito calculo!")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Nosso Terceiro Modelo — SVM no Titanic</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Aplicando o que aprendemos nos dados reais."</p></div></div></div>


### De volta ao Titanic

Agora que entendemos a intuição, a margem e o kernel, aplicamos o SVM
ao nosso problema de classificação de sobrevivência.

Como o Titanic tem features mistas (algumas lineares, outras não),
testaremos os dois kernels principais: `linear` e `rbf`.


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ── SVM Linear ────────────────────────────────────────────────────────────────
svm_linear = SVC(kernel="linear", C=1.0, probability=True, random_state=42)
svm_linear.fit(X_treino_sc, y_treino)

# ── SVM RBF ───────────────────────────────────────────────────────────────────
svm_rbf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)
svm_rbf.fit(X_treino_sc, y_treino)

print("SVMs treinados no Titanic!")
print()
print(f"SVM Linear — Vetores de Suporte: {svm_linear.n_support_}")
print(f"  Total: {sum(svm_linear.n_support_)} "
      f"({sum(svm_linear.n_support_)/len(X_treino_sc):.1%} do treino)")
print()
print(f"SVM RBF — Vetores de Suporte: {svm_rbf.n_support_}")
print(f"  Total: {sum(svm_rbf.n_support_)} "
      f"({sum(svm_rbf.n_support_)/len(X_treino_sc):.1%} do treino)")
print()
print("Nota: mais vetores de suporte = fronteira mais complexa.")


In [ ]:
# Avaliação dos dois SVMs
y_pred_lin = svm_linear.predict(X_teste_sc)
y_pred_rbf = svm_rbf.predict(X_teste_sc)

print("=" * 55)
print("SVM LINEAR — Relatório completo")
print("=" * 55)
print(classification_report(y_teste, y_pred_lin,
      target_names=["Nao Sobreviveu","Sobreviveu"]))

print("=" * 55)
print("SVM RBF — Relatório completo")
print("=" * 55)
print(classification_report(y_teste, y_pred_rbf,
      target_names=["Nao Sobreviveu","Sobreviveu"]))


In [ ]:
# Matrizes de confusão lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("SVM no Titanic — Matrizes de Confusão", fontweight="bold")

for ax, y_pred, kernel_nome, cmap in [
    (axes[0], y_pred_lin, "Linear", "Blues"),
    (axes[1], y_pred_rbf, "RBF",    "Purples"),
]:
    acc = accuracy_score(y_teste, y_pred)
    f1  = f1_score(y_teste, y_pred)
    cm  = confusion_matrix(y_teste, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["Nao Sobrev.","Sobreviveu"])         .plot(ax=ax, colorbar=False, cmap=cmap)
    ax.set_title(f"SVM {kernel_nome} Acurácia: {acc:.1%}  |  F1: {f1:.4f}",
                 fontweight="bold")

plt.tight_layout()
plt.savefig("aula07_svm_titanic_cm.png", dpi=110, bbox_inches="tight")
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Compare os dois SVMs (Linear e RBF) e responda: (a) qual teve maior acurácia? (b) qual teve maior Recall para a classe 'Sobreviveu'? (c) para o contexto do Titanic, qual dos dois você escolheria e por quê? Registre antes de ver o gabarito.</span></div>

*✏️ (a) Maior acurácia: SVM `???`*

*✏️ (b) Maior Recall (Sobreviveu): SVM `???`*

*✏️ (c) Escolheria: `???` porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# from sklearn.metrics import recall_score, precision_score
#
# for nome, yp in [("SVM Linear", y_pred_lin), ("SVM RBF", y_pred_rbf)]:
#     print(f"{nome}:")
#     print(f"  Acuracia:  {accuracy_score(y_teste, yp):.4f}")
#     print(f"  F1:        {f1_score(y_teste, yp):.4f}")
#     print(f"  Recall:    {recall_score(y_teste, yp):.4f}")
#     print(f"  Precisao:  {precision_score(y_teste, yp):.4f}")
#     print()
#
# print("Interpretacao:")
# print("  O Titanic tem uma fronteira parcialmente nao-linear (interacoes")
# print("  entre classe social, genero e idade).")
# print("  O SVM RBF normalmente captura essas interacoes melhor.")
# print()
# print("  Para o contexto do Titanic:")
# print("  Se priorizamos Recall (encontrar todos sobreviventes): usar o de maior Recall.")
# print("  Se priorizamos balanco geral: comparar F1.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Ajustando C e gamma — Encontrando os Melhores Hiperparâmetros</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Nenhum hiperparâmetro é bom por padrão — precisamos testar."</p></div></div></div>


### Grid Search — Testando combinações sistematicamente

Em vez de testar um valor de C de cada vez, podemos usar o
**GridSearchCV** para testar todas as combinações de valores
e encontrar a melhor usando validação cruzada.


In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

# ── Grid Search no SVM RBF ────────────────────────────────────────────────────
grade = {
    "C":     [0.1, 1.0, 10.0, 100.0],
    "gamma": ["scale", 0.001, 0.01, 0.1],
}

grid_svm = GridSearchCV(
    SVC(kernel="rbf", probability=True, random_state=42),
    grade,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=0,
)
grid_svm.fit(X_treino_sc, y_treino)

print("GRID SEARCH — SVM RBF")
print("=" * 50)
print(f"Melhores hiperparâmetros:  {grid_svm.best_params_}")
print(f"Melhor F1 (CV 5-fold):     {grid_svm.best_score_:.4f}")
print()

# Top 5 combinações
resultados = pd.DataFrame(grid_svm.cv_results_)
top5 = (resultados[["param_C","param_gamma","mean_test_score","std_test_score"]]
        .sort_values("mean_test_score", ascending=False)
        .head(5))
top5.columns = ["C", "gamma", "F1 médio", "F1 std"]
print("Top 5 combinações:")
print(top5.round(4).to_string(index=False))


In [ ]:
# Heatmap de Grid Search — visualizando o espaço de hiperparâmetros
resultados_pivot = resultados.pivot_table(
    index="param_C",
    columns="param_gamma",
    values="mean_test_score"
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(
    resultados_pivot,
    annot=True, fmt=".3f",
    cmap="YlOrRd",
    vmin=resultados_pivot.values.min(),
    vmax=resultados_pivot.values.max(),
    ax=ax,
    linewidths=0.5,
    annot_kws={"size": 10},
)
ax.set_title("Grid Search — F1 por C e gamma (SVM RBF) Vermelho escuro = melhor combinação",
             fontweight="bold")
ax.set_xlabel("gamma"); ax.set_ylabel("C")

# Marcando o melhor
melhor_C     = grid_svm.best_params_["C"]
melhor_gamma = grid_svm.best_params_["gamma"]
print(f"Melhor: C={melhor_C}, gamma={melhor_gamma}")

plt.tight_layout()
plt.savefig("aula07_gridsearch_heatmap.png", dpi=110, bbox_inches="tight")
plt.show()


In [ ]:
# Avaliando o SVM otimizado
svm_otimizado = grid_svm.best_estimator_
y_pred_otim   = svm_otimizado.predict(X_teste_sc)
y_prob_otim   = svm_otimizado.predict_proba(X_teste_sc)[:,1]

print("SVM RBF OTIMIZADO (Grid Search)")
print("=" * 50)
print(f"Parâmetros: C={melhor_C}, gamma={melhor_gamma}")
print()
print(classification_report(y_teste, y_pred_otim,
      target_names=["Nao Sobreviveu","Sobreviveu"]))

print(f"AUC-ROC: {roc_auc_score(y_teste, y_prob_otim):.4f}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — Interprete o heatmap do Grid Search: (a) em qual região (C alto ou baixo, gamma alto ou baixo) o modelo performa melhor? (b) o que acontece com F1 quando C=100 e gamma=0.1 ao mesmo tempo? (c) o Grid Search melhorou o F1 em relação ao SVM com parâmetros padrão (C=1, gamma=scale)?</span></div>

*✏️ (a) Melhor região: C `???` e gamma `???`*

*✏️ (b) C=100 e gamma=0.1: `???` — porque: `???`*

*✏️ (c) Melhoria do Grid Search: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# # Comparando SVM padrao vs otimizado
# svm_padrao = SVC(kernel="rbf", C=1.0, gamma="scale",
#                  probability=True, random_state=42).fit(X_treino_sc, y_treino)
# y_pred_pad = svm_padrao.predict(X_teste_sc)
#
# f1_padrao = f1_score(y_teste, y_pred_pad)
# f1_otim   = f1_score(y_teste, y_pred_otim)
#
# print("Comparacao SVM padrao vs otimizado:")
# print(f"  SVM RBF (C=1, gamma=scale): F1 = {f1_padrao:.4f}")
# print(f"  SVM RBF (Grid Search):      F1 = {f1_otim:.4f}")
# print(f"  Ganho: {(f1_otim - f1_padrao)*100:+.2f} pontos percentuais")
# print()
# print("Interpretacao do heatmap:")
# print("  - gamma muito alto + C muito alto = overfitting (F1 cai)")
# print("  - gamma muito baixo = fronteira suave demais = underfitting")
# print("  - O ponto ideal geralmente esta numa regiao intermediaria")
# print("  - 'scale' e uma boa opcao automatica para gamma na maioria dos casos")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 6</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Placar Geral — KNN vs Reg. Logística vs SVM</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Três modelos, um dataset, métricas iguais — quem vence no Titanic?"</p></div></div></div>


### Comparação objetiva

Chegamos ao fim da trilogia de modelos clássicos.
Vamos comparar os três lado a lado com as mesmas métricas,
nos mesmos dados de teste.


In [ ]:
from sklearn.metrics import roc_curve, precision_score, recall_score

# Coletando resultados de todos os modelos
modelos = {
    "KNN (K=7)":         (knn_ref,      X_teste_sc),
    "Reg. Logistica":    (lr_ref,       X_teste_sc),
    "SVM Linear":        (svm_linear,   X_teste_sc),
    "SVM RBF (padrao)":  (svm_rbf,      X_teste_sc),
    "SVM RBF (otimiz.)": (svm_otimizado,X_teste_sc),
}

tabela = []
for nome, (modelo, Xte) in modelos.items():
    yp  = modelo.predict(Xte)
    ypr = modelo.predict_proba(Xte)[:,1]
    tabela.append({
        "Modelo":      nome,
        "Acuracia":    accuracy_score(y_teste, yp),
        "Precisao":    precision_score(y_teste, yp),
        "Recall":      recall_score(y_teste, yp),
        "F1":          f1_score(y_teste, yp),
        "AUC-ROC":     roc_auc_score(y_teste, ypr),
    })

df_tabela = pd.DataFrame(tabela).set_index("Modelo")

print("PLACAR GERAL — Todos os Modelos")
print("=" * 72)
print(df_tabela.round(4).to_string())
print()
print("Melhor por métrica:")
for col in df_tabela.columns:
    vencedor = df_tabela[col].idxmax()
    val      = df_tabela[col].max()
    print(f"  {col:<12} -> {vencedor}  ({val:.4f})")


In [ ]:
# Painel visual completo
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.suptitle("Comparação Final — KNN vs Regressão Logística vs SVM  |  Titanic",
             fontsize=13, fontweight="bold")

CORES = ["#a8d8ea","#0f3460","#f0a500","#e94560","#2ecc71"]
nomes = list(df_tabela.index)

# Gráfico 1: métricas por modelo (barras agrupadas)
metricas_plot = ["Acuracia","F1","AUC-ROC"]
x_pos = np.arange(len(nomes))
largura = 0.25
cores_m = ["#0f3460","#e94560","#f0a500"]

for i, (met, cor) in enumerate(zip(metricas_plot, cores_m)):
    axes[0].bar(x_pos + i*largura, df_tabela[met],
                width=largura, color=cor, edgecolor="white",
                alpha=0.85, label=met)

axes[0].set_xticks(x_pos + largura)
axes[0].set_xticklabels(nomes, rotation=25, ha="right", fontsize=8)
axes[0].set_ylabel("Valor")
axes[0].set_title("Métricas por Modelo", fontweight="bold")
axes[0].legend(fontsize=9); axes[0].set_ylim(0.65, 0.95)

# Gráfico 2: Precisão vs Recall (scatter)
for i, (nome, row) in enumerate(df_tabela.iterrows()):
    axes[1].scatter(row["Recall"], row["Precisao"],
                    s=180, color=CORES[i], zorder=5,
                    edgecolors="white", linewidth=1.5)
    axes[1].annotate(nome, (row["Recall"], row["Precisao"]),
                     textcoords="offset points",
                     xytext=(6, 4), fontsize=8)

axes[1].plot([0,1],[0,1], "k--", alpha=0.3, linewidth=1)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precisão")
axes[1].set_title("Precisão vs Recall (ideal: canto superior direito)",
                  fontweight="bold")
axes[1].set_xlim(0.5, 1.05); axes[1].set_ylim(0.55, 1.05)

# Gráfico 3: Curvas ROC
for i, (nome, (modelo, Xte)) in enumerate(modelos.items()):
    ypr = modelo.predict_proba(Xte)[:,1]
    fpr, tpr, _ = roc_curve(y_teste, ypr)
    auc = roc_auc_score(y_teste, ypr)
    lw  = 3 if "otimiz" in nome else 1.8
    axes[2].plot(fpr, tpr, color=CORES[i], linewidth=lw,
                 label=f"{nome} (AUC={auc:.3f})", alpha=0.85)

axes[2].plot([0,1],[0,1],"k--",linewidth=1,alpha=0.4,label="Aleatório")
axes[2].fill_between([0,1],[0,1], alpha=0.03, color="gray")
axes[2].set_xlabel("FPR (Falso Positivo)")
axes[2].set_ylabel("TPR (Recall)")
axes[2].set_title("Curvas ROC (mais alto e esquerda = melhor)",
                  fontweight="bold")
axes[2].legend(fontsize=7, loc="lower right")

plt.tight_layout()
plt.savefig("aula07_placar_final.png", dpi=110, bbox_inches="tight")
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 5 — Analise a tabela e os gráficos e responda: (a) qual modelo você colocaria em produção para o problema do Titanic? (b) a otimização do SVM (Grid Search) valeu a pena? A melhoria foi significativa? (c) para um projeto em que o tempo de predição importa muito, qual modelo você descartaria?</span></div>

*✏️ (a) Colocaria em produção: `???` porque: `???`*

*✏️ (b) Otimização valeu? `???` — diferença de F1: `???`*

*✏️ (c) Descartaria para predição rápida: `???` porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 5 (descomente para ver) ───────────────────────────────
# print("Gabarito — Analise final:")
# print()
# print("(a) Escolha para producao:")
# print("    Nao ha resposta unica — depende dos requisitos:")
# print()
# print("    Se INTERPRETABILIDADE importa (explicar para o passageiro/seguradora):")
# print("    -> Regressao Logistica: coeficientes claros, rapida, confiavel")
# print()
# print("    Se PERFORMANCE maxima importa:")
# print("    -> SVM RBF otimizado: tende a ter melhor AUC em dados com fronteiras nao-lineares")
# print()
# print("    Se VELOCIDADE de predicao importa (milhoes de predicoes/segundo):")
# print("    -> Regressao Logistica ou SVM Linear (nao precisa calcular distancias)")
# print()
# print("(b) Otimizacao SVM:")
# f1_pad_num = f1_score(y_teste, svm_rbf.predict(X_teste_sc))
# f1_oti_num = f1_score(y_teste, svm_otimizado.predict(X_teste_sc))
# print(f"    SVM RBF padrao:    F1 = {f1_pad_num:.4f}")
# print(f"    SVM RBF otimizado: F1 = {f1_oti_num:.4f}")
# print(f"    Diferenca: {(f1_oti_num - f1_pad_num)*100:+.2f} pontos")
# print("    Em datasets pequenos, o ganho costuma ser pequeno.")
# print("    Em datasets grandes e complexos, pode ser muito relevante.")
# print()
# print("(c) Descartar para predicao rapida: KNN")
# print("    O KNN precisa calcular distancias para TODOS os pontos de treino")
# print("    a cada predicao. Com 1 milhao de amostras, isso e inviavel.")


---

## Guia de Seleção — Os Três Algoritmos

| Critério | KNN | Reg. Logística | SVM |
|----------|-----|---------------|-----|
| **Interpretabilidade** | Baixa | Alta | Média |
| **Fronteiras não-lineares** | Sim (local) | Não | Sim (com kernel) |
| **Velocidade de treino** | Instantâneo | Rápido | Médio-lento |
| **Velocidade de predição** | Lento | Muito rápido | Rápido |
| **Datasets grandes** | Ruim | Ótimo | Moderado |
| **Normalização obrigatória** | Sim | Recomendada | Sim |
| **Hiperparâmetro principal** | K | C | C, gamma, kernel |
| **Produz probabilidades** | Sim (estimada) | Sim (direto) | Sim (com `probability=True`) |
| **Ponto forte no Titanic** | Padrões locais | Interpretação | Fronteiras complexas |

---

## Checklist — O que você sabe fazer agora

| Habilidade | Praticada hoje? |
|------------|----------------|
| Explicar a intuição de margem máxima e vetores de suporte | ☐ |
| Diferenciar margem rígida de suave e o papel do parâmetro C | ☐ |
| Entender por que o Kernel Trick permite fronteiras não-lineares | ☐ |
| Treinar SVM linear e RBF com scikit-learn | ☐ |
| Realizar Grid Search para otimizar C e gamma | ☐ |
| Interpretar o heatmap do Grid Search | ☐ |
| Comparar três modelos com métricas, curva ROC e análise de erros | ☐ |
| Escolher o modelo certo baseado nos requisitos do problema | ☐ |

---


## Reflexão final

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">Escreva em suas próprias palavras: (1) o que é um vetor de suporte e por que ele tem esse nome?, (2) por que o Kernel Trick é considerado 'elegante'?, (3) em qual situação real você usaria SVM em vez de Regressão Logística?</span></div>


**✏️ Minha reflexão:**

1. Vetor de suporte é: *...* — tem esse nome porque: *...*

2. O Kernel Trick é elegante porque: *...*

3. Usaria SVM em vez de Regressão Logística quando: *...*


---

## O que vem a seguir?

```
Concluídos:  KNN ✅   Regressão Logística ✅   SVM ✅
Próximos:    Árvores de Decisão  →  Random Forest  →  Gradient Boosting
```

Na próxima aula você vai conhecer as **Árvores de Decisão** — o algoritmo
mais intuitivo do curso. Ao contrário do SVM (que é uma caixa-preta elegante),
a árvore **mostra exatamente como tomou cada decisão**, em formato de
perguntas encadeadas que qualquer pessoa consegue seguir.

---

## Referências

- Scikit-Learn SVM: https://scikit-learn.org/stable/modules/svm.html
- Cortes, C. & Vapnik, V. (1995). *Support-Vector Networks*. Machine Learning, 20, 273–297.
- Géron, A. (2019). *Hands-on Machine Learning*, Cap. 5. O'Reilly.
- Kernel Trick visual: https://youtu.be/3liCbRZPrZA
